# GEAP Agent Evaluation — Interactive Demo Notebook

This notebook walks the **Quality Flywheel** end-to-end with 100% coverage of Google's
[Gemini Enterprise Agent Platform → Optimize → Evaluation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/agent-evaluation) docs.

Each section links the doc page it covers and calls the matching demo step. Where the SDK
returns a rich result, we call `.show()` for interactive tables (a feature of the
[view-results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results) page).

See also: [`docs/evaluation_demo.md`](../../../docs/evaluation_demo.md) and the coverage
matrix in [`docs/eval_operations.md` §0](../../../docs/eval_operations.md).


## Setup


In [1]:
import vertexai
from src.config import AGENT_ENGINE_ID
from src.eval.demo import steps

client = steps.make_client()            # Agent Platform SDK client (Vertex)
RESOURCE = steps.resolve_resource(AGENT_ENGINE_ID)
AGENT = 'coordinator_agent'
print('client ready:', client is not None, '| agent resource:', RESOURCE)


client ready: True | agent resource: projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024


/home/admin_jwortz_altostrat_com/geap-tour/src/eval/demo/steps.py:32: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  return Client(project=GCP_PROJECT_ID, location=GCP_REGION)


## Phase 1 — Design: the Metric Registry
📖 [manage-metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics)

Three metric types: predefined rubric (`types.RubricMetric.*`), custom LLM-as-judge
(`types.LLMMetric`), and custom deterministic code (`types.CodeExecutionMetric`), plus a
reference-based Exact Match. Register once, reuse across offline runs and online monitors.


In [2]:
from src.eval import metric_registry as mr
print('custom metrics:', [getattr(m, 'name', type(m).__name__) for m in mr.custom_metrics()])
# Register them in the Metric Registry (writes to your project):
# mr.register_all(client)
steps.register_metrics(client)['catalog']


custom metrics: ['policy_compliance', 'geap_tool_use', 'policy_limit_exact', 'exact_match']


{'single_turn_rubric': ['FINAL_RESPONSE_QUALITY',
  'HALLUCINATION',
  'TOOL_USE_QUALITY',
  'SAFETY'],
 'multi_turn_rubric': ['MULTI_TURN_TASK_SUCCESS',
  'MULTI_TURN_TOOL_USE_QUALITY',
  'MULTI_TURN_TRAJECTORY_QUALITY'],
 'custom': ['policy_compliance',
  'geap_tool_use',
  'policy_limit_exact',
  'exact_match'],
 'exact_match_reference_based': True,
 'code_metric': True}

## Phase 2a — Rapid evaluation + `result.show()`
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

A quick pointwise LLM-judge run over a few prompts. `result.show()` renders the aggregate
and per-case tables inline.


In [3]:
res = steps.rapid_eval(client, RESOURCE)
raw = res.get('raw')
if raw is not None:
    raw.show()   # interactive summary + per-case scores
{k: v for k, v in res.items() if k != 'raw'}


Running one-time eval on projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024...
  Dataset: 5 prompts
  Metrics: final_response_quality, instruction_following, general_quality
  Running inference (querying the deployed agent)...
  Running evaluation...


20:26:49 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
20:26:50 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
Computing Metrics for Evaluation Dataset: 100%|██████████| 15/15 [00:37<00:00,  2.49s/it]


=== Evaluation Results ===
  final_response_quality_v1: mean=0.56 (total=5, errors=0)
  instruction_following_v1: mean=0.76 (total=5, errors=0)
  general_quality_v1: mean=0.82 (total=5, errors=0)


{'step': 2,
 'title': 'Rapid evaluation (client.evals.evaluate)',
 'doc': 'https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents',
 'status': 'ok',
 'metrics': [{'metric': 'final_response_quality_v1', 'mean': 0.560000008},
  {'metric': 'instruction_following_v1', 'mean': 0.76},
  {'metric': 'general_quality_v1', 'mean': 0.82333334}]}

## Phase 2b — Test-case / regression batch
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

Runs the per-agent regression suite against the deployed engine (this can take a few
minutes — it runs inference then a scored evaluation run).


In [ ]:
steps.testcase_eval(AGENT_ENGINE_ID, AGENT)


## Phase 2c — Simulated scenario evaluation
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

Auto-generates conversation scenarios (`generate_conversation_scenarios`: a starting prompt + a
hidden conversation plan, grounded by `environment_context`), runs the deployed agent on each
scenario's opening turn, and scores the responses with rubric autoraters. (Multi-turn
`run_inference` is unavailable on the pinned SDK — see the note in `src/eval/simulated_eval.py` —
so we score the opening turn of each generated scenario.)
</cell id="cell09">


In [ ]:
steps.simulate(RESOURCE, AGENT, scenario_count=3, max_turns=4)


## Phase 2d — Environment simulation (resilience)
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

Intercept tool calls to inject mocked data and simulated failures (HTTP 503) — test how the
agent recovers, without touching production backends.


In [7]:
steps.environment_simulation()


Environment simulation demo — agent_name='travel_agent'
In production, replace each MCP toolset on the agent with an
intercepted version, then run the simulated-user eval:
    live_tools = {t.name: t for t in agent.tools}         # MCP tools
    wrapped    = wrap_tools(live_tools, mocks=..., error_every=3)
    # ...attach `wrapped` to the agent, then:
    run_simulated_eval(agent_resource, agent_name=..., multi_turn=True)

Wrapped 3 tools: ['check_expense_policy', 'search_flights', 'search_hotels']
Error injection: every 3 call(s)
------------------------------------------------------------------------
  call 1: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 'FL002', 'route': 'SFO->JFK', 'price': 510, 'airline': 'Delta'}]
  call 2: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 'FL002', 'route': 'SFO->JFK', 'price': 510, 'airline'

{'step': 5,
 'title': 'Environment simulation (mock tools + injected 503s)',
 'doc': 'https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated',
 'status': 'ok',
 'summary': {'agent_name': 'travel_agent',
  'agent_resource': None,
  'tools_wrapped': ['check_expense_policy', 'search_flights', 'search_hotels'],
  'total_calls': 6,
  'mock_hits': 5,
  'injected_errors': 1,
  'error_every': 3}}

## Phase 2e — Offline evaluation over historical traces/sessions
📖 [evaluate-offline](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-offline)

Score **already-recorded** traces retroactively (no new inference). Reads gen_ai OTel
events from BigQuery, falling back to a bundled fixture.


In [6]:
steps.offline_eval(client, AGENT)


Offline trace evaluation (coordinator_agent)
  Agent resource: projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024
  BigQuery unavailable (400 GET https://bigquery.googleapis.com/bigquery/v2/projects/wortz-project-352116/queries/b9676894-a469-4a17-b98f-2cc5e38e2349?maxResults=0&location=US&prettyPrint=false: Unrecognized name: json_payload at [11:22]

Location: US
Job ID: b9676894-a469-4a17-b98f-2cc5e38e2349
); falling back to bundled fixture.
  Source: fixture /home/admin_jwortz_altostrat_com/geap-tour/src/eval/sample_traces.jsonl (8 traces)
  Scoring 8 historical traces with 3 metrics...


Computing Metrics for Evaluation Dataset: 100%|██████████| 24/24 [00:38<00:00,  1.61s/it]


  Per-metric results:
    [FAIL] final_response_quality_v1: score=1.72/5 (raw=0.344, total=8, errors=0, threshold=3.0)
    [PASS] hallucination_v1: score=5.00/5 (raw=1.0, total=8, errors=0, threshold=3.0)
    [PASS] safety_v1: score=4.38/5 (raw=0.875, total=8, errors=0, threshold=3.0)


{'step': 6,
 'title': 'Offline evaluation over historical traces/sessions',
 'doc': 'https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-offline',
 'status': 'ok',
 'result': {'status': 'ok',
  'source': 'fixture',
  'agent_name': 'coordinator_agent',
  'hours_back': 24,
  'record_count': 8,
  'score_threshold': 3.0,
  'timestamp': '2026-07-28T22:25:38.789765',
  'metrics': {'final_response_quality_v1': {'raw_mean': 0.34375,
    'score': 1.71875,
    'num_cases_total': 8,
    'num_cases_error': 0,
    'status': 'FAIL'},
   'hallucination_v1': {'raw_mean': 1.0,
    'score': 5.0,
    'num_cases_total': 8,
    'num_cases_error': 0,
    'status': 'PASS'},
   'safety_v1': {'raw_mean': 0.875,
    'score': 4.375,
    'num_cases_total': 8,
    'num_cases_error': 0,
    'status': 'PASS'}}}}

## Phase 3 — Continuous evaluation with Online Monitors
📖 [evaluate-online](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online)

Online Monitors asynchronously score live production traces on a ~10-minute loop and export
scores to Cloud Logging + Cloud Monitoring. Create/verify with
`python -m src.eval.setup_online_evaluators create`.


In [8]:
steps.online_monitors(do_setup=False)


{'step': 7,
 'title': 'Continuous evaluation with Online Monitors',
 'doc': 'https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online',
 'status': 'ok',
 'note': 'Online Monitors asynchronously score live traces on a ~10-min loop (Query -> Evaluate -> Report), exporting to Cloud Logging + Cloud Monitoring. Create/verify with: python -m src.eval.setup_online_evaluators create|verify'}

## Phase 4 — Optimize agent prompts (close the flywheel) 🔁
📖 [optimize-agent](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/optimize-agent)

The finale: feed the eval signal back into a **live GEPA optimization** that auto-tunes the
coordinator's root instruction against its eval set, then returns the improved instruction. This
feature-detects the documented `client.optimizer.optimize(...)` and falls back to the ADK
`GEPARootAgentPromptOptimizer`. It's a real run against the deployed agent (a few minutes; the MCP
tool servers must be reachable), bounded by `max_metric_calls` for the demo — this is what closes the
Quality Flywheel: **Evaluate → Analyze → Optimize → re-evaluate**.
</cell id="cell19">


In [ ]:
# Live finale: run a bounded GEPA optimization of the coordinator's instruction against its eval set.
# This actually re-tunes the prompt — it runs the deployed agent, so the MCP tool servers must be
# reachable. Expect a few minutes; the optimized instruction is returned in result['result'].
result = steps.optimize(client, run=True, max_metric_calls=30)
result


## Quality-drift alerts
📖 [quality-alerts](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/quality-alerts)

Export a Cloud Monitoring alert policy on `aiplatform.googleapis.com/online_evaluator/scores`
and apply it with `gcloud monitoring policies create --policy-from-file=...`.


In [10]:
steps.quality_alerts()


✓ Alert policy YAML written: src/eval/policies/quality_drift_policy.yaml
  Metric: aiplatform.googleapis.com/online_evaluator/scores (evaluation_metric_name=task_success) < 0.8
  Apply: gcloud monitoring policies create --policy-from-file=src/eval/policies/quality_drift_policy.yaml


{'step': 10,
 'title': 'Quality-drift alerts (Cloud Monitoring policy)',
 'doc': 'https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/quality-alerts',
 'status': 'ok',
 'policy_file': 'src/eval/policies/quality_drift_policy.yaml'}

## Recap

You just ran the full Quality Flywheel: **Design → Execution → Scoring → Refinement**,
covering all nine Optimize → Evaluation doc pages. For the headless orchestrator + JSON
report, run:

```bash
uv run python -m src.eval.demo.full_eval_demo --agent-id $AGENT_ENGINE_ID --emit-json eval_outputs/demo/full_demo.json
```
